# Mapping the Potential Destructive Power of Wildfires Using Machine Learning

## Module 5: *Feature Interaction Analysis*

---
### Contents  
- 1. *Load Data*
- 2. *Split and Scale Data*
- 2. *Feature Interaction Analysis*
- 6. *Export Data*
---
### Notes
---
### Inputs
- `final.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `X`,`y` - for future modeling
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

# Function to print a custom format correlation heatmap
from src.plot_utils import correlation_map

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions

# A space saving function to rank interactions
from src.data_utils import rank_interactions_by_correlation

# Function to calculate dryness index and return a dataframe
from src.data_utils import calculate_dryness_index

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize

# Modeling prep
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import PolynomialFeatures

# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

## 1. Load Data

In [ ]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/final.csv")

In [6]:
final

,Sample_ID,Date,Sample_Elevation,Region_ID,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Avg Air Temp (F),Avg Rel Hum (%),Avg Wind Speed (mph),Avg Soil Temp (F),Season,Total Population,density,Mean Income,Target
0,1,2018-01-01,36.609302,7,0.05,0.02,230.0,14.3,55.4,96.0,3.1,56.8,0,3269973,777.337917,111241,0
1,33,2018-01-01,1096.443481,5,0.10,0.00,260.0,4.1,51.9,31.0,4.1,53.8,0,441257,161.331803,111731,0
2,144,2018-01-01,1039.047485,1,0.06,0.00,228.0,8.4,48.9,71.0,3.4,47.0,0,64896,22.000807,74237,0
3,32,2018-01-01,288.821625,5,0.15,0.00,449.0,9.5,59.2,55.0,3.9,61.8,0,441257,161.331803,111731,0
4,145,2018-01-01,1615.276733,2,0.05,0.00,222.0,8.8,47.1,80.0,1.8,45.9,0,207172,126.597656,79233,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,52,2025-01-23,925.951294,6,0.10,0.00,347.0,1.5,43.7,16.0,4.9,40.6,0,2195611,109.468892,85327,0
447776,124,2025-01-23,473.016693,1,0.07,0.00,276.0,6.2,45.2,60.0,2.5,46.0,0,89108,25.413394,74102,0
447777,51,2025-01-23,1046.279663,6,0.10,0.00,347.0,1.5,43.7,16.0,4.9,40.6,0,2195611,109.468892,85327,0
447778,50,2025-01-23,956.628479,4,0.15,0.00,333.0,1.2,39.9,14.0,1.8,49.7,0,913820,112.374445,75161,0


---

## 2. Split and Scale Data

In [ ]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target']

y = final['Target']
X = final.drop(columns=text_columns)
details = final[text_columns]

Scale all datasets and save back to dataframe format. MinMax Scaler used because majority of values are > 0

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

---

## 3. Interaction Correlation analysis

For all independent variables create a new dataframe containing every possible interaction of 2nd degree

In [18]:
inter_X = create_2nd_degree_interactions(X_scaled)

Combine interaction terms with single variables for correlation analysis

In [19]:
inter_X_combined = pd.concat([X_scaled, inter_X], axis=1)

In [20]:
correlation_results = rank_interactions_by_correlation(inter_X_combined, y)

C:\Users\dusti\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\dusti\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
correlation_results.head(20)

,Feature,Correlation
0,ETo (in) x 2-Year Avg Fires,0.135693
1,Avg Air Temp (F) x Avg Air Temp (F) 7 Day Avg,0.127724
2,Sol Rad (Ly/day) x 2-Year Avg Fires,0.126044
3,Avg Air Temp (F),0.125936
4,Avg Air Temp (F) x Dryness,0.124976
5,ETo (in) x Season,0.124823
6,ETo (in) x Avg Vap Pres (mBars),0.124725
7,Avg Air Temp (F) x Avg Soil Temp (F),0.123603
8,Avg Air Temp (F) x 2-Year Avg Fires,0.121435
9,Avg Air Temp (F) 7 Day Avg,0.121307


Keep top 20 results

In [23]:
top_20 = inter_X_combined[keep]

# Recompute correlation matrix
corr_matrix = top_20.corr()

to_drop = set()
cols = corr_matrix.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if corr_matrix.iloc[i, j] > 0.9:
            col_to_drop = cols[j]  # drop the later column in the pair
            to_drop.add(col_to_drop)

df_reduced = top_20.drop(columns=list(to_drop))

df_reduced

,ETo (in) x 2-Year Avg Fires,Avg Air Temp (F) x Avg Air Temp (F) 7 Day Avg,ETo (in) x Season,ETo (in) x Avg Vap Pres (mBars),Avg Vap Pres (mBars) x Avg Air Temp (F),ETo (in) x Avg Air Temp (F),Avg Wind Speed (mph) x 2-Year Avg Fires,Avg Vap Pres (mBars) x Season,Sample_Elevation x 2-Year Avg Fires
0,0.011620,0.234487,0.0,0.027666,0.146209,0.048861,0.007988,0.0,0.002577
342,0.013944,0.255860,0.0,0.035056,0.162920,0.061873,0.009019,0.0,0.002577
396,0.004648,0.266750,0.0,0.012382,0.176421,0.021077,0.007473,0.0,0.002577
546,0.013944,0.268802,0.0,0.031806,0.150811,0.063127,0.009276,0.0,0.002577
844,0.013944,0.270298,0.0,0.034360,0.162920,0.063127,0.011595,0.0,0.002577
...,...,...,...,...,...,...,...,...,...
446933,0.000000,0.053170,0.0,0.005417,0.018371,0.020067,0.000000,0.0,0.000000
447129,0.000000,0.049860,0.0,0.004488,0.014376,0.018952,0.000000,0.0,0.000000
447392,0.000000,0.051714,0.0,0.005611,0.015116,0.024909,0.000000,0.0,0.000000
447561,0.000000,0.051871,0.0,0.005998,0.016893,0.026042,0.000000,0.0,0.000000


In [24]:
X = df_reduced

---

## 4. Export Data

In [25]:
X.to_csv('../data/processed/X.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
